In [1]:
def dsp(xi, xf, f_draft, f_sketch, f_proof):
    yi = f_draft(xi)
    zf = f_sketch(yi, xi, xf)
    yf = f_proof(xf, zf)
    return yf

In [1]:
import time

In [35]:
class Checker(object):
    """A modified version of the Draft, Sketch, Prove proof-checking client.
    (https://github.com/albertqjiang/draft_sketch_prove/blob/main/autoformalization/checker.py)

    This checker supports Isabelle2022 via the new version of PISA
    (https://albertqjiang.github.io/Portal-to-ISAbelle/).

    It supports checking a miniF2F-style proof via `check`.

    Finally, it replaces `sledgehammer` with a call to `normalhammer`.
    """
    def __init__(self, working_dir, isa_path, theory_file, port=9000):
        sys.path.append(os.environ['PISA_PATH'])
        try:
            from pisa_client import initialise_env
            self.initialise_env = initialise_env
        except:
            print("Set $PISA_PATH to /yourpath/to/Portal-to-ISAbelle/src/main/python")

        self.working_dir = working_dir
        self.isa_path = isa_path
        self.theory_file = theory_file
        self.port = port

    def _initialize(self):
        env = self.initialise_env(
            self.port,
            isa_path=self.isa_path,
            theory_file=self.theory_file,
            working_directory=self.working_dir
        )
        print("Environment initialized:", env)
        return env

    def _exit(self, env):
        try:
            env.post('exit')
        except:
            print("env.post('exit') timed out")
            pass
        os.system("ps aux | grep Isabelle | awk '{print $2}' | xargs kill -9 > /dev/null 2>&1")
        os.system("ps aux | grep poly | awk '{print $2}' | xargs kill -9 > /dev/null 2>&1")

    def _parse_output(self, obs):
        """Parse the sledgehammer output, otherwise return an empty string"""
        if '<hammer>' in obs:
            output = obs.split('<hammer>')[0]
        else:
            output = ''    
        return output
    """
    def _run_step(self, step, i, tls_name, env):
        obs, reward, done, metadata = env.step_to_top_level_state(
            action=step,
            tls_name=tls_name,
            new_name='default_%d' % i
        )
        error = None
        if 'error:' in obs or 'Step error' in obs or 'Unknown error' in obs:
            error = obs
        return obs, reward, done, metadata, error
    

    def _run_step(self, step, i, tls_name, env):
        print(f"Executing step {i}: {step}")
        try:
            obs, reward, done, metadata = env.step_to_top_level_state(
                action=step,
                tls_name=tls_name,
                new_name=f'default_{i}'
            )
            print(f"Step {i} observation: {obs}")
            return obs, reward, done, metadata, None
        except Exception as e:
            print(f"Step {i} failed: {e}")
            return '', 0, False, None, str(e)
    """
    
    def _run_step(self, step, i, tls_name, env):
        print(f"Executing step {i}: {step}")
        try:
            # Log sledgehammer calls
            if "sledgehammer" in step:
                print(f"Calling sledgehammer for step {i}: {step}")
            obs, reward, done, metadata = env.step_to_top_level_state(
                action=step,
                tls_name=tls_name,
                new_name=f'default_{i}'
            )
            print(f"Step {i} observation: {obs}")
            return obs, reward, done, metadata, None
        except Exception as e:
            print(f"Step {i} failed: {e}")
            return '', 0, False, None, str(e)

    
    def _run_sledgehammer(self, step, i, tls_name, env):
        # First try heuristics
        for heuristic in ['by auto', 'by simp', 'by blast', 'by fastforce', 'by force', 'by eval', 'by presburger', 'by sos', 'by arith', 'by linarith', 'by (auto simp: field_simps)']:
            step_ = step.replace('normalhammer', heuristic)
            obs, reward, done, metadata, error = self._run_step(step_, i, tls_name, env)
            if error is None:
                obs = '%s <hammer> %s' % (heuristic, obs)
                return obs, reward, done, metadata, error
        # Try sledgehammer
        out = self._run_step(step, i, tls_name, env)
        return out

    def check(self, statement_and_proof):
        # Initialize environment
        env = self._initialize()
        env.initialise()

        # Wrap and parse theorem
        theory = Checker.wrap_theorem(statement_and_proof)
    
        steps = Checker.get_parsed(env, theory)

        result = self._check(env, steps)
        return result

    def _check(self, env, steps):
        done = False
        reason = ''
        success = False
        step_results = []
        tls_name = 'default'
        for i, step in enumerate(steps):
            try:
                time0 = time.time()
                if 'normalhammer' in step:
                    obs, reward, done, metadata, error = self._run_sledgehammer(step, i, tls_name, env)
                else:
                    obs, reward, done, metadata, error = self._run_step(step, i, tls_name, env)
                step_time = time.time() - time0
                step_results.append(dict(index=i, step=step, output=self._parse_output(obs), step_time=step_time))
                if error is not None:
                    reason = error
                    success = False
                    done = False
                    break
            except:
                # Timeout - end the proof attempt
                success = False
                done = False
                reason = 'timeout (%d)' % len(step_results)
                step_results.append(dict(index=i, step=step, output=''))
                break

            # Change when successful
            tls_name = 'default_%d' % i

        if done and reward == 1.0:
            success = True

        result = {
            'success': success,
            'reason': reason,
            'num_steps': len(steps),
            'last_step': len(step_results),
            'step_results': step_results,
            'theorem_and_proof': self.reconstruct(step_results) if success else ''
        }
        # Exit environment
        self._exit(env)
        return result
    
    @staticmethod
    def reconstruct(step_results):
        steps = []
        for step_result in step_results[1:]:
            if step_result['output'] != '':
                steps.append(step_result['output'].strip())
            else:
                steps.append(step_result['step'].strip())
        theorem_and_proof = '\n'.join(steps)
        return theorem_and_proof

    @staticmethod
    def wrap_theorem(theorem):
        return 'theory Interactive imports HOL.HOL Complex_Main "HOL-Library.Code_Target_Numeral" "HOL-Library.Sum_of_Squares" "Symmetric_Polynomials.Vieta" "HOL-Computational_Algebra.Computational_Algebra" "HOL-Number_Theory.Number_Theory" \n begin\n%s' % theorem

    """
    @staticmethod
    def get_parsed(env, theory, tls_name='default'):
        # HACK: the parsing doesn't work well with `normalhammer`, so we replace
        # all hammer calls with sorry, then replace sorry to normalhammer after parsing.
        theory = theory.replace('sledgehammer', 'sorry')
        theory = theory.replace('normalhammer', 'sorry')

        print("Parsing theory:", theory)

        steps = env.post(f"<parse text> ${theory}")

        print("Parsed steps:", steps)
        steps = steps.split('<SEP>')
        steps = [s for s in steps if s.strip() != '']
        # remove weird '$' step and whitespace steps
        steps = [s for s in steps if s != '$' and s.strip() != '']
        steps = [s.replace('sorry', 'normalhammer') for s in steps]
        return steps
    
    def get_parsed(env, theory, tls_name='default'): #Works for the simple one
    # Parse the theory
        print("Parsing theory:", theory)
        steps = env.post(f"<parse text> ${theory}")
        print("Parsed steps:", steps)
    
        # Split steps and filter out invalid entries
        steps = steps.split('<SEP>')
        steps = [s.strip() for s in steps if s.strip() and s != '$']  # Exclude empty and `$` steps
        steps = [s.replace('sorry', 'normalhammer') for s in steps]
    
        return steps
    
    @staticmethod
    def get_parsed(env, theory, tls_name='default'): #Works for the simple one
        print("Parsing theory:", theory)
        # Parse the theory using the PISA environment
        raw_steps = env.post(f"<parse text> ${theory}")
        print("Raw parsed steps:", raw_steps)
    
        # Split into individual steps
        steps = raw_steps.split('<SEP>')
        steps = [s.strip() for s in steps if s.strip()]  # Remove empty or whitespace-only steps
    
        # Filter out invalid entries
        steps = [s for s in steps if s != '$']  # Exclude standalone `$`
        
        # Process "then" logic
        processed_steps = []
        for i, step in enumerate(steps):
            if step.lower() == "then":
                # Skip "then" if there's no valid context
                if i == 0 or steps[i - 1].startswith("proof"):
                    print(f"Skipping standalone 'then' at step {i}: {step}")
                    continue
            processed_steps.append(step)
    
        # Debugging the filtered steps
        print("Processed steps:", processed_steps)
        return processed_steps

    """

    @staticmethod
    def get_parsed(env, theory, tls_name='default'):
        print("Parsing theory:", theory)
        # Parse the theory using the PISA environment
        raw_steps = env.post(f"<parse text> ${theory}")
        print("Raw parsed steps:", raw_steps)
    
        # Split into individual steps
        steps = raw_steps.split('<SEP>')
        steps = [s.strip() for s in steps if s.strip()]  # Remove empty or whitespace-only steps
    
        # Filter out invalid entries
        steps = [s for s in steps if s != '$']  # Exclude standalone `$`
        
        # Process "then" logic
        processed_steps = []
        for i, step in enumerate(steps):
            if step.lower() == "then":
                # Skip "then" if there's no valid context
                if i == 0 or steps[i - 1].startswith("proof"):
                    print(f"Skipping standalone 'then' at step {i}: {step}")
                    continue
            processed_steps.append(step)
    
        # Debugging the filtered steps
        print("Processed steps:", processed_steps)
        return processed_steps


In [4]:
import sys
import os
sys.path.append('../')
os.environ['PISA_PATH'] = '/home/balaji/Desktop/Nesy_Contd/Portal-to-ISAbelle/src/main/python'

#import dsp_utils 

checker = Checker(
    working_dir='/home/balaji/Desktop/Isabelle2022/src/HOL/Examples',
    isa_path='/home/balaji/Desktop/Isabelle2022',
    theory_file='/home/balaji/Desktop/Isabelle2022/src/HOL/Examples/Interactive.thy',
    port=9000
)


In [5]:
from pisa_client import initialise_env
help(initialise_env)

Help on function initialise_env in module pisa_client:

initialise_env(port, isa_path, theory_file, working_directory, debug=False)
    # Initialize the Isabelle environment



In [39]:
import os
import sys

# Check PISA_PATH
pisa_path = os.environ.get('PISA_PATH', None)
if not pisa_path:
    print("PISA_PATH is not set.")
else:
    print(f"PISA_PATH is set to: {pisa_path}")
    if not os.path.exists(pisa_path):
        print(f"Error: The path {pisa_path} does not exist.")
    else:
        print(f"Files in PISA_PATH: {os.listdir(pisa_path)}")

# Check Isabelle paths
isa_path = '/home/balaji/Desktop/Isabelle2022'
working_dir = '/home/balaji/Desktop/Isabelle2022/src/HOL/Examples'
theory_file = '/home/balaji/Desktop/Isabelle2022/src/HOL/Examples/Interactive.thy'

print("\nChecking Isabelle paths:")
for path in [isa_path, working_dir, theory_file]:
    if not os.path.exists(path):
        print(f"Error: The path {path} does not exist.")
    else:
        print(f"{path} exists.")


PISA_PATH is set to: /home/balaji/Desktop/Nesy_Contd/Portal-to-ISAbelle/src/main/python
Files in PISA_PATH: ['server_pb2_grpc.py', 'data_extraction', '.ipynb_checkpoints', 'conjecturing_parsing', '__pycache__', '.idea', 'pisa_client.py', 'test_client.py', 'utils', 'server_pb2.py', 'legacy']

Checking Isabelle paths:
/home/balaji/Desktop/Isabelle2022 exists.
/home/balaji/Desktop/Isabelle2022/src/HOL/Examples exists.
/home/balaji/Desktop/Isabelle2022/src/HOL/Examples/Interactive.thy exists.


In [29]:
from pisa_client import initialise_env

# Initialize the environment with required parameters
env = initialise_env(
    port=9000,  # Replace with your port if not 9000
    isa_path='/home/balaji/Desktop/Isabelle2022',  # Path to Isabelle installation
    theory_file='/home/balaji/Desktop/Isabelle2022/src/HOL/Examples/Interactive.thy',  # Path to your Isabelle theory file
    working_directory='/home/balaji/Desktop/Isabelle2022/src/HOL/Examples'  # Path to Isabelle working directory
)

# Initialize the PISA-Isabelle environment
try:
    env.initialise()
    print("PISA-Isabelle communication successful.")
except Exception as e:
    print(f"Error during PISA-Isabelle initialization: {e}")

# Cleanly exit the environment
try:
    env.post('exit')
    print("Environment exited successfully.")
except Exception as e:
    print(f"Error during environment exit: {e}")


Initializing Isabelle...
Isa Path: /home/balaji/Desktop/Isabelle2022
Working Directory: /home/balaji/Desktop/Isabelle2022/src/HOL/Examples
Theory File: /home/balaji/Desktop/Isabelle2022/src/HOL/Examples/Interactive.thy
----------Path to Isabelle source----------
/home/balaji/Desktop/Isabelle2022
----------Path to Isabelle working directory----------
/home/balaji/Desktop/Isabelle2022/src/HOL/Examples
----------Path to Isabelle theory file----------
/home/balaji/Desktop/Isabelle2022/src/HOL/Examples/Interactive.thy
PISA-Isabelle communication successful.
Environment exited successfully.


In [49]:
theorem_and_sledgehammer_proof = """theorem simple_theorem: "(0::nat) + 1 = 1"
  by simp
"""

result = checker.check(theorem_and_sledgehammer_proof)

print("\n==== Success: %s" % result['success'])
print("--- Complete proof:\n%s" % result['theorem_and_proof'])

Initializing Isabelle...
Isa Path: /home/balaji/Desktop/Isabelle2022
Working Directory: /home/balaji/Desktop/Isabelle2022/src/HOL/Examples
Theory File: /home/balaji/Desktop/Isabelle2022/src/HOL/Examples/Interactive.thy
----------Path to Isabelle source----------
/home/balaji/Desktop/Isabelle2022
----------Path to Isabelle working directory----------
/home/balaji/Desktop/Isabelle2022/src/HOL/Examples
----------Path to Isabelle theory file----------
/home/balaji/Desktop/Isabelle2022/src/HOL/Examples/Interactive.thy
Environment initialized: <pisa_client.PisaEnv object at 0x7f92372a4880>
Parsing theory: theory Interactive imports HOL.HOL Complex_Main "HOL-Library.Code_Target_Numeral" "HOL-Library.Sum_of_Squares" "Symmetric_Polynomials.Vieta" "HOL-Computational_Algebra.Computational_Algebra" "HOL-Number_Theory.Number_Theory" 
 begin
theorem simple_theorem: "(0::nat) + 1 = 1"
  by simp

Raw parsed steps: $<SEP>theory Interactive imports HOL.HOL Complex_Main "HOL-Library.Code_Target_Numeral" 

In [26]:
theorem_and_sledgehammer_proof = """theorem simple_theorem: "(0::nat) + 1 = 1"
  sledgehammer
"""

result = checker.check(theorem_and_sledgehammer_proof)

print("\n==== Success: %s" % result['success'])
print("--- Complete proof:\n%s" % result['theorem_and_proof'])

Initializing Isabelle...
Isa Path: /home/balaji/Desktop/Isabelle2022
Working Directory: /home/balaji/Desktop/Isabelle2022/src/HOL/Examples
Theory File: /home/balaji/Desktop/Isabelle2022/src/HOL/Examples/Interactive.thy
----------Path to Isabelle source----------
/home/balaji/Desktop/Isabelle2022
----------Path to Isabelle working directory----------
/home/balaji/Desktop/Isabelle2022/src/HOL/Examples
----------Path to Isabelle theory file----------
/home/balaji/Desktop/Isabelle2022/src/HOL/Examples/Interactive.thy
Environment initialized: <pisa_client.PisaEnv object at 0x7f14f4042eb0>
Parsing theory: theory Interactive imports HOL.HOL Complex_Main "HOL-Library.Code_Target_Numeral" "HOL-Library.Sum_of_Squares" "Symmetric_Polynomials.Vieta" "HOL-Computational_Algebra.Computational_Algebra" "HOL-Number_Theory.Number_Theory" 
 begin
theorem simple_theorem: "(0::nat) + 1 = 1"
  sledgehammer

Raw parsed steps: $<SEP>theory Interactive imports HOL.HOL Complex_Main "HOL-Library.Code_Target_Nume

In [30]:
class Checker(object):
    """A modified version of the Draft, Sketch, Prove proof-checking client.
    (https://github.com/albertqjiang/draft_sketch_prove/blob/main/autoformalization/checker.py)

    This checker supports Isabelle2022 via the new version of PISA
    (https://albertqjiang.github.io/Portal-to-ISAbelle/).

    It supports checking a miniF2F-style proof via `check`.

    Finally, it replaces `sledgehammer` with a call to `normalhammer`.
    """
    def __init__(self, working_dir, isa_path, theory_file, port=9000):
        sys.path.append(os.environ.get('PISA_PATH', ''))
        try:
            from pisa_client import initialise_env
            self.initialise_env = initialise_env
        except ImportError:
            print("Set $PISA_PATH to /yourpath/to/Portal-to-ISAbelle/src/main/python")

        self.working_dir = working_dir
        self.isa_path = isa_path
        self.theory_file = theory_file
        self.port = port

    def _initialize(self):
        env = self.initialise_env(
            self.port,
            isa_path=self.isa_path,
            theory_file=self.theory_file,
            working_directory=self.working_dir
        )
        print("Environment initialized:", env)
        self._test_sledgehammer(env)
        return env

    def _exit(self, env):
        try:
            env.post('exit')
        except Exception:
            print("env.post('exit') timed out")
        os.system("ps aux | grep Isabelle | awk '{print $2}' | xargs kill -9 > /dev/null 2>&1")
        os.system("ps aux | grep poly | awk '{print $2}' | xargs kill -9 > /dev/null 2>&1")

    def _test_sledgehammer(self, env):
        """Test if sledgehammer is available in the Isabelle environment."""
        try:
            test_theory = """
            theory SledgehammerTest imports Main begin
            theorem test_theorem: "(0::nat) + 1 = 1"
            proof -
              sledgehammer
            qed
            end
            """
            print("Testing sledgehammer availability...")
            result = env.post(f"<parse text> ${test_theory}")
            print(f"Sledgehammer test result: {result}")
        except Exception as e:
            print(f"Sledgehammer test failed: {e}")

    def _parse_output(self, obs):
        """Parse the sledgehammer output, otherwise return an empty string."""
        if '<hammer>' in obs:
            return obs.split('<hammer>')[0]
        return ''

    def _run_step(self, step, i, tls_name, env):
        print(f"Executing step {i}: {step}")
        try:
            obs, reward, done, metadata = env.step_to_top_level_state(
                action=step,
                tls_name=tls_name,
                new_name=f'default_{i}'
            )
            print(f"Step {i} observation: {obs}")
            return obs, reward, done, metadata, None
        except Exception as e:
            print(f"Step {i} failed with error: {e}")
            # Retry with sledgehammer if applicable
            if "by" in step:
                step = step.replace("by simp", "sledgehammer").replace("by auto", "sledgehammer")
                print(f"Retrying step {i} with sledgehammer: {step}")
                return self._run_step(step, i, tls_name, env)
            return '', 0, False, None, str(e)

    def _run_sledgehammer(self, step, i, tls_name, env):
        if "sledgehammer" in step:
            print(f"Directly calling sledgehammer for step {i}: {step}")
            return self._run_step(step, i, tls_name, env)

        for heuristic in [
            'by auto', 'by simp', 'by blast', 'by fastforce', 
            'by force', 'by eval', 'by presburger', 'by sos', 
            'by arith', 'by linarith', 'by (auto simp: field_simps)'
        ]:
            step_ = step.replace('normalhammer', heuristic)
            obs, reward, done, metadata, error = self._run_step(step_, i, tls_name, env)
            if error is None:
                obs = f'{heuristic} <hammer> {obs}'
                return obs, reward, done, metadata, error

        print(f"Falling back to sledgehammer for step {i}")
        return self._run_step(step.replace("normalhammer", "sledgehammer"), i, tls_name, env)

    def check(self, statement_and_proof):
        env = self._initialize()
        env.initialise()

        theory = Checker.wrap_theorem(statement_and_proof)
        steps = Checker.get_parsed(env, theory)

        result = self._check(env, steps)
        return result

    def _check(self, env, steps):
        done = False
        reason = ''
        success = False
        step_results = []
        tls_name = 'default'

        for i, step in enumerate(steps):
            try:
                time0 = time.time()
                if 'normalhammer' in step or 'sledgehammer' in step:
                    obs, reward, done, metadata, error = self._run_sledgehammer(step, i, tls_name, env)
                else:
                    obs, reward, done, metadata, error = self._run_step(step, i, tls_name, env)

                step_time = time.time() - time0
                step_results.append(dict(index=i, step=step, output=self._parse_output(obs), step_time=step_time))

                if error:
                    reason = error
                    print(f"Step {i} failed with error: {error}")
                    break
            except Exception as e:
                reason = f"Timeout at step {i}: {e}"
                print(reason)
                step_results.append(dict(index=i, step=step, output=''))
                break

            tls_name = f'default_{i}'

        success = done and reward == 1.0

        result = {
            'success': success,
            'reason': reason,
            'num_steps': len(steps),
            'last_step': len(step_results),
            'step_results': step_results,
            'theorem_and_proof': self.reconstruct(step_results) if success else ''
        }
        self._exit(env)
        return result

    @staticmethod
    def reconstruct(step_results):
        return '\n'.join(
            step_result['output'].strip() if step_result['output'] else step_result['step'].strip()
            for step_result in step_results[1:]
        )

    @staticmethod
    def wrap_theorem(theorem):
        return (
            'theory Interactive imports HOL.HOL Complex_Main '
            '"HOL-Library.Code_Target_Numeral" "HOL-Library.Sum_of_Squares" '
            '"Symmetric_Polynomials.Vieta" "HOL-Computational_Algebra.Computational_Algebra" '
            '"HOL-Number_Theory.Number_Theory" \n begin\n%s' % theorem
        )

    @staticmethod
    def get_parsed(env, theory, tls_name='default'):
        print("Parsing theory:", theory)
        raw_steps = env.post(f"<parse text> ${theory}")
        print("Raw parsed steps:", raw_steps)

        steps = [s.strip() for s in raw_steps.split('<SEP>') if s.strip() and s != '$']

        processed_steps = []
        for i, step in enumerate(steps):
            if step.lower() == "then" and (i == 0 or steps[i - 1].startswith("proof")):
                print(f"Skipping standalone 'then' at step {i}: {step}")
                continue
            processed_steps.append(step)

        print("Processed steps:", processed_steps)
        return processed_steps


In [40]:
theorem_and_sledgehammer_proof ="""
theorem amc12a_2008_p8:
  fixes x y::real
  assumes h0: "0 < x \<and> 0 < y"
    and h1: "y^3 = 1"
    and h2: "6 * x^2 = 2 * (6 * y^2)"
  shows "x^3 = 2 * sqrt 2"
  using assms
  by (smt (verit, best) mult_cancel_left2 one_power2 
      power2_eq_square power2_le_imp_le 
      power2_sum power3_eq_cube power_Suc_less 
      power_commutes power_gt1_lemma real_le_lsqrt 
      real_le_rsqrt)
"""

result = checker.check(theorem_and_sledgehammer_proof)

print("\n==== Success: %s" % result['success'])
print("--- Complete proof:\n%s" % result['theorem_and_proof'])

Initializing Isabelle...
Isa Path: /home/balaji/Desktop/Isabelle2022
Working Directory: /home/balaji/Desktop/Isabelle2022/src/HOL/Examples
Theory File: /home/balaji/Desktop/Isabelle2022/src/HOL/Examples/Interactive.thy
----------Path to Isabelle source----------
/home/balaji/Desktop/Isabelle2022
----------Path to Isabelle working directory----------
/home/balaji/Desktop/Isabelle2022/src/HOL/Examples
----------Path to Isabelle theory file----------
/home/balaji/Desktop/Isabelle2022/src/HOL/Examples/Interactive.thy
Environment initialized: <pisa_client.PisaEnv object at 0x7f14f4069340>
Parsing theory: theory Interactive imports HOL.HOL Complex_Main "HOL-Library.Code_Target_Numeral" "HOL-Library.Sum_of_Squares" "Symmetric_Polynomials.Vieta" "HOL-Computational_Algebra.Computational_Algebra" "HOL-Number_Theory.Number_Theory" 
 begin

theorem amc12a_2008_p8:
  fixes x y::real
  assumes h0: "0 < x \<and> 0 < y"
    and h1: "y^3 = 1"
    and h2: "6 * x^2 = 2 * (6 * y^2)"
  shows "x^3 = 2 * sqr

In [38]:
# Initialize Checker
checker = Checker(
    working_dir="/home/balaji/Desktop/Isabelle2022/src/HOL/Examples",
    isa_path="/home/balaji/Desktop/Isabelle2022",
    theory_file="/home/balaji/Desktop/Isabelle2022/src/HOL/Examples/Interactive.thy"
)

# Simple proof statement
statement_and_proof = """
theorem amc12b_2002_p6:
  fixes a b :: real
  assumes "a \<noteq> 0 \<and> b \<noteq> 0"
      and "\<forall> x. x^2 + a * x + b = (x - a) * (x - b)"
    shows " a = 1 \<and> b = -2"
  using assms 
  by (metis (no_types, opaque_lifting) Groups.mult_ac(2) Rings.ring_distribs(2) add.inverse_inverse 
      add_uminus_conv_diff diff_0 diff_numeral_special(10) mult.left_neutral 
      mult_cancel_left power2_eq_square right_minus_eq)
"""

# Check the proof
result = checker.check(statement_and_proof)

# Output results
if result['success']:
    print("Proof succeeded!")
else:
    print("Proof failed.")
    for step in result['steps']:
        print(f"Step {step['index']} -> {step['step']}:\n{step['output']}")


Initializing Isabelle...
Isa Path: /home/balaji/Desktop/Isabelle2022
Working Directory: /home/balaji/Desktop/Isabelle2022/src/HOL/Examples
Theory File: /home/balaji/Desktop/Isabelle2022/src/HOL/Examples/Interactive.thy
----------Path to Isabelle source----------
/home/balaji/Desktop/Isabelle2022
----------Path to Isabelle working directory----------
/home/balaji/Desktop/Isabelle2022/src/HOL/Examples
----------Path to Isabelle theory file----------
/home/balaji/Desktop/Isabelle2022/src/HOL/Examples/Interactive.thy
Environment initialized: <pisa_client.PisaEnv object at 0x7f14f4069430>
Parsing theory: theory Interactive imports HOL.HOL Complex_Main "HOL-Library.Code_Target_Numeral" "HOL-Library.Sum_of_Squares" "Symmetric_Polynomials.Vieta" "HOL-Computational_Algebra.Computational_Algebra" "HOL-Number_Theory.Number_Theory" 
 begin

theorem amc12b_2002_p6:
  fixes a b :: real
  assumes "a \<noteq> 0 \<and> b \<noteq> 0"
      and "\<forall> x. x^2 + a * x + b = (x - a) * (x - b)"
    shows